# DBRepo Starter Notebook

The API documentation for the DBRepo, together with coding examples, can be found under [https://www.ifs.tuwien.ac.at/infrastructures/dbrepo/dbrepo_v1.13.3.pdf](https://www.ifs.tuwien.ac.at/infrastructures/dbrepo/dbrepo_v1.13.3.pdf).




## Initialize the REST Client

In [1]:
import pandas as pd
from dbrepo.RestClient import RestClient
from getpass import getpass
import dbrepo.api.dto as dto
from dbrepo.api.dto import IdentifierType, CreateIdentifierTitle, CreateIdentifierCreator, CreateIdentifierDescription

In [3]:
# Constants for clarity and easy modification
DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
CONTAINER_ID = "6cfb3b8e-1792-4e46-871a-f3d103527203"
DATABASE_DISPLAY_NAME = "SDCC Traffic Flow Experiment 2022"

# User Credentials
# Using a more descriptive name than just 'username'
username = "e11705340"
user_password = getpass(f"Enter password for {username}: ")

client = RestClient(
    endpoint=DBREPO_ENDPOINT, 
    username=username, 
    password=user_password
)


In [42]:
containers = client.get_containers()
print(containers)
if len(containers) == 0:
    print("No containers found. Please create a container first.")
if len(containers) == 1:
    old = CONTAINER_ID
    CONTAINER_ID = containers[0].id
    print(f"container_id has been updated {old} --> {CONTAINER_ID}")
else:
    print("More than one container found. Please specify the container_id manually.")

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]
container_id has been updated 6cfb3b8e-1792-4e46-871a-f3d103527203 --> 6cfb3b8e-1792-4e46-871a-f3d103527203


## Create the Traffic Database

In [5]:
# create_database always creates a new database.
# To update an existing database, provide the database_id.
# DATABASE_ID = None
DATABASE_ID = "2de50b61-ac24-4484-8117-dc0fe9dc1b7c"

# 2. Create the Database (Technical Initialization)
if DATABASE_ID is None:
    database = client.create_database(
        name=DATABASE_DISPLAY_NAME,
        container_id=CONTAINER_ID,
        is_public=True,
        is_schema_public=True
    )
print(f"Database created with ID: {DATABASE_ID}")

Database created with ID: 2de50b61-ac24-4484-8117-dc0fe9dc1b7c


In [57]:
# Create the Identifier (Metadata & Citability)
# This is where you specify the original publisher and license to ensure
# semantic correctness and proper attribution.
client.create_identifier(
    database_id=DATABASE_ID,
    type=IdentifierType.DATABASE,
    creators=[
        CreateIdentifierCreator(
            creator_name="Group 5"
        )
    ],
    titles=[
        CreateIdentifierTitle(
        title=DATABASE_DISPLAY_NAME
        )
    ],
    descriptions=[
        CreateIdentifierDescription(
            description="Relational 3NF representation of SDCC Traffic Flow Data (Jan-June 2022). "
                        "Original data sourced from the SDCC Open Data Portal.",
            language=dto.Language.EN,
            type=dto.DescriptionType.ABSTRACT
        )
    ],
    # Set the original publisher of the traffic data
    publisher="South Dublin County Council",

    publication_year=2022,

    related_identifiers=[
        dto.CreateRelatedIdentifier(
            value="https://data.smartdublin.ie/dataset/traffic-flow-data-jan-to-june-2022-sdcc1",
            type=dto.RelatedIdentifierType.URL,
            relation=dto.RelatedIdentifierRelation.CITES
        )
    ],
    licenses=[
        dto.License(
            identifier="CC-BY",
            uri="http://www.opendefinition.org/licenses/cc-by",
            description=""
        )
    ]
)

print("Metadata identifier created successfully. The database is now citable.")

MalformedError: Failed to create identifier: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}

In [6]:
# check the identifiers
client.get_identifiers(DATABASE_ID)

[]

## Create the Tables and upload the data:

In [9]:
# --- 1. Load the Measurements (CSV) ---
# CSV_DATA_URL = "https://data-sdublincoco.opendata.arcgis.com/api/download/v1/items/ce994c07d66e4ce582c4d608f339fcd9/csv?layers=0"
# df_measurements = pd.read_csv(CSV_DATA_URL, on_bad_lines='warn')
df_raw = pd.read_csv("../data/raw/Traffic_Flow_Data_Jan_to_June_2022_SDCC.csv", on_bad_lines='warn')

print("Number of rows:", len(df_raw))
print("Available columns csv:", df_raw.columns.tolist())
print("Number of unique sites:", len(df_raw["site"].unique()))
print("Number of unique dates:", len(df_raw["date"].unique()))


Number of rows: 1048575
Available columns csv: ['site', 'day', 'date', 'start_time', 'end_time', 'flow', 'flow_pc', 'cong', 'cong_pc', 'dsat', 'dsat_pc', 'ObjectId']
Number of unique sites: 61
Number of unique dates: 181


### CALENDAR (Dimension Table)

In [18]:
# --- 2. CALENDAR (Dimension Table) ---
df_calendar = df_raw[['date', 'day']].drop_duplicates().reset_index(drop=True)
df_calendar.columns = ['date', 'day_of_week']

df_calendar["day_of_week"] = df_calendar["day_of_week"].astype(str)
df_calendar = df_calendar.set_index("date")

print("\nCalendar Table Sample:\n", df_calendar.head())


Calendar Table Sample:
            day_of_week
date                  
04/01/2022          TU
01/01/2022          SA
03/01/2022          MO
02/01/2022          SU
05/01/2022          WE


In [19]:
print(len(df_raw["date"].unique()), "unique dates found in raw data.")
print(len(df_calendar), "unique dates found.")

181 unique dates found in raw data.
181 unique dates found.


In [21]:
calendar_table = client.create_table(
    database_id=DATABASE_ID,
    name="Calendar",
    is_public=True,
    is_schema_public=True,
    dataframe=df_calendar,
    description="Table to handle date-to-day mapping.",
    with_data=True,
)

print(f"Created table: {calendar_table.name}")

2026-05-16 13:01:33,192 root         WARNING default to 'text' for column date and type <class 'numpy.dtype'>
2026-05-16 13:01:33,195 root         WARNING default to 'text' for column day_of_week and type <class 'numpy.dtype'>
Created table: Calendar


In [20]:
# Delete the table if it already exists (needed for retry)
# !!!only uncomment if you want to delete the table!!!
# client.delete_table(DATABASE_ID, "<table-id>")

### TRAFFIC_SITES (Dimension Table)

In [56]:
# --- 2. TRAFFIC_SITES (Dimension Table) ---
df_sites = df_raw[['site']].drop_duplicates().reset_index(drop=True)
df_sites.columns = ['site_id']
df_sites = df_sites.set_index('site_id')

print("Sites Table\n", df_sites.head())
print(df_sites.info())

Sites Table
 Empty DataFrame
Columns: []
Index: [N01111A, N01111C, N01111D, N01111M, N01111N]
<class 'pandas.DataFrame'>
Index: 61 entries, N01111A to N03121D
Empty DataFrame
None


In [113]:
traffic_sites_table = client.create_table(
    database_id=DATABASE_ID,
    name="TrafficSites",
    is_public=True,
    is_schema_public=True,
    dataframe=df_sites,
    description="Contains data for physical sensor locations.",
    with_data=True,
)

print(f"Created table: {traffic_sites_table.name}")

2026-05-15 13:51:51,539 root         WARNING default to 'text' for column site_id and type <class 'numpy.dtype'>
Created table: TrafficSites


### TRAFFIC_MEASUREMENTS (Fact Table)

In [52]:
# --- 4. TRAFFIC_MEASUREMENTS (Fact Table) ---
# Hier speichern wir die Messdaten und verknüpfen sie über Fremdschlüssel (site_id, date)
df_measurements = df_raw[[
    'site', 'date', 'end_time', 'flow', 'flow_pc',
    'cong', 'cong_pc', 'dsat', 'dsat_pc'
]].copy()

# Spaltennamen an das ER-Modell anpassen
df_measurements.rename(columns={'site': 'site_id'}, inplace=True)

df_measurements['start_time'] = (pd.to_datetime(
    df_measurements['end_time'].str.replace('24:00', '00:00'),
    format='%H:%M'
) - pd.Timedelta(minutes=15)).dt.strftime('%H:%M')

# Primärschlüssel (observation_id) generieren
df_measurements.insert(0, 'observation_id', range(1, len(df_measurements) + 1))
df_measurements = df_measurements.set_index(['observation_id'])

print("\nMeasurements Table Sample:\n", df_measurements.head())
print(df_measurements.info())


Measurements Table Sample:
                 site_id        date end_time  flow  flow_pc  cong  cong_pc  \
observation_id                                                               
1               N01111A  04/01/2022    03:15    13      100     0      100   
2               N01111A  04/01/2022    03:30    10      100     0      100   
3               N01111A  04/01/2022    03:45     0      100     0      100   
4               N01111A  04/01/2022    04:00     9      100     0      100   
5               N01111A  04/01/2022    04:15     0      100     0      100   

                dsat  dsat_pc start_time  
observation_id                            
1                  0        0      03:00  
2                  0        0      03:15  
3                  0        0      03:30  
4                  0        0      03:45  
5                  0        0      04:00  
<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 1 to 1048575
Data columns (total 10 columns):
 #   Column      Non-

In [55]:
traffic_measurements_table = client.create_table(
    database_id=DATABASE_ID,
    name="TrafficMeasurements",
    is_public=True,
    is_schema_public=True,
    dataframe=df_measurements,
    description="Core fact table containing 15-minute traffic flow and congestion metrics.",
    with_data=True,
)

print(f"Created table: {traffic_measurements_table.name}")
print("Data upload complete!")

2026-05-16 13:50:02,118 root         WARNING default to 'text' for column site_id and type <class 'numpy.dtype'>
2026-05-16 13:50:02,192 root         WARNING default to 'text' for column date and type <class 'numpy.dtype'>
2026-05-16 13:50:02,304 root         WARNING default to 'text' for column end_time and type <class 'numpy.dtype'>
2026-05-16 13:50:03,518 root         WARNING default to 'text' for column start_time and type <class 'numpy.dtype'>
Created table: TrafficMeasurements
Data upload complete!


In [54]:
# Delete the table if it already exists (needed for retry)
# !!!only uncomment if you want to delete the table!!!
# client.delete_table(DATABASE_ID, "<table-id>")